# 002_structured_data_completed.ipynb — Claude en Vertex AI

Extracción de datos forzando la herramienta article_summary.

La primera celda configura el cliente y las rutas. Las llamadas requieren un proyecto y credenciales Google reales. [Configuración](../README.md).

In [ ]:
# Cliente real de Vertex AI: Application Default Credentials (ADC).
import os
from pathlib import Path

from anthropic import AnthropicVertex
from dotenv import load_dotenv

repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "GOOGLE" / "README.md").is_file()
)
google_root = repo_root / "GOOGLE"
load_dotenv(google_root / ".env", override=False)
exercise_dir = google_root / "ToolsBuildingWithApi"
os.chdir(exercise_dir)

project_id = os.getenv("GOOGLE_CLOUD_PROJECT")
if not project_id:
    raise ValueError("Configura GOOGLE_CLOUD_PROJECT en GOOGLE/.env.")
client = AnthropicVertex(
    project_id=project_id,
    region=os.getenv("CLOUD_ML_REGION", "global"),
)
model = os.getenv("VERTEX_MODEL_ID", "claude-sonnet-4-6")


In [ ]:
# Conserva todos los bloques de las respuestas, incluidas herramientas y thinking.
from anthropic.types import Message


def message_content(message):
    content = message.content if isinstance(message, Message) else message
    if isinstance(content, str):
        if not content.strip():
            raise ValueError("El texto del mensaje no puede estar vacío.")
    elif not isinstance(content, list) or not content:
        raise ValueError("El mensaje necesita bloques de contenido.")
    else:
        for block in content:
            if isinstance(block, dict) and block.get("type") == "text" and not block.get("text", "").strip():
                raise ValueError("Los bloques de texto no pueden estar vacíos.")
    return content


def add_user_message(messages, message):
    messages.append({"role": "user", "content": message_content(message)})


def add_assistant_message(messages, message):
    messages.append({"role": "assistant", "content": message_content(message)})


def text_from_message(message):
    return "\n".join(block.text for block in message.content if block.type == "text")


def chat(
    messages,
    system=None,
    stop_sequences=None,
    tools=None,
    tool_choice=None,
    max_tokens=4000,
):
    params = {"model": model, "max_tokens": max_tokens, "messages": messages}
    if stop_sequences:
        params["stop_sequences"] = list(stop_sequences)
    if system:
        params["system"] = system
    if tools:
        params["tools"] = list(tools)
        if tool_choice:
            params["tool_choice"] = tool_choice
    message = client.messages.create(**params)
    return message


In [ ]:
article_summary_schema = {
    "name": "article_summary",
    "description": "Creates a summary of an article with its key insights. Use this tool when you need to generate a structured summary of an article, research paper, or any textual content. The tool requires the article's title, author name, and a list of the most important insights or takeaways from the content. Each insight should be a concise statement capturing a significant point from the article.",
    "input_schema": {
        "type": "object",
        "properties": {
            "title": {
                "type": "string",
                "description": "The title of the article being summarized.",
            },
            "author": {
                "type": "string",
                "description": "The name of the author who wrote the article.",
            },
            "key_insights": {
                "type": "array",
                "items": {"type": "string"},
                "description": "A list of the most important takeaways or insights from the article. Each insight should be a complete, concise statement.",
            },
        },
        "required": ["title", "author", "key_insights"],
    },
}

In [ ]:
messages = []
add_user_message(
    messages,
    """
    Write a one-paragraph scholarly article about computer science. 
    Include a title and author name.
    """,
)
response = chat(messages)
text_from_message(response)

In [ ]:
messages = []

add_user_message(messages, text_from_message(response))
response = chat(
    messages,
    tools=[article_summary_schema],
    tool_choice={"type": "tool", "name": "article_summary"},
)
next(block.input for block in response.content if block.type == "tool_use" and block.name == "article_summary")